<a href="https://colab.research.google.com/github/Canahmetozguven/app_market_analyse/blob/main/notebooks/bl_research_lab_v10_large_universe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Türkiye yatırım research notebook'u — large universe v10

Bu sürüm geniş evren deneyi için hazırlandı:

- BIST hisseleri
- USDTRY, EURTRY
- ALTIN_TRY, GUMUS_TRY
- TEFAS fonları

Dürüst not:
- burada **güncel BIST100 üyelerini canlı doğrulayamıyorum**
- bu yüzden notebook geniş evreni destekleyecek şekilde kuruldu
- `BIST_TICKERS` ve `TEFAS_FUND_TICKERS` hücrelerini senin doğruladığın listeyle güncellemek gerekir


In [ ]:
!pip -q install openbb yfinance PyPortfolioOpt plotly pandas numpy scipy scikit-learn openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.4/379.4 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.8/99.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.7/97.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.5/125.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.1/66.1 

In [ ]:
import warnings, os, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px

from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.hierarchical_portfolio import HRPOpt
from pypfopt.black_litterman import BlackLittermanModel
from pypfopt.efficient_frontier import EfficientFrontier


## 1) Universe config

`BIST_TICKERS` ve `TEFAS_FUND_TICKERS` doğrudan bu hücreden yönetilir.


In [ ]:
BIST_TICKERS = ['AEFES.IS', 'AGHOL.IS', 'AKBNK.IS', 'AKFYE.IS', 'AKSA.IS', 'AKSEN.IS', 'ALARK.IS', 'ARCLK.IS', 'ASELS.IS', 'ASTOR.IS', 'BIMAS.IS', 'BOBET.IS', 'CCOLA.IS', 'CIMSA.IS', 'CWENE.IS', 'DOAS.IS', 'DOHOL.IS', 'ECILC.IS', 'EGEEN.IS', 'EKGYO.IS', 'ENERY.IS', 'ENJSA.IS', 'ENKAI.IS', 'EREGL.IS', 'EUPWR.IS', 'FROTO.IS', 'GARAN.IS', 'GESAN.IS', 'GUBRF.IS', 'HALKB.IS', 'HEKTS.IS', 'ISCTR.IS', 'ISMEN.IS', 'KAYSE.IS', 'KCAER.IS', 'KCHOL.IS', 'KLSER.IS', 'KOZAA.IS', 'KOZAL.IS', 'KRDMD.IS', 'MAVI.IS', 'MGROS.IS', 'MIATK.IS', 'ODAS.IS', 'OTKAR.IS', 'OYAKC.IS', 'PETKM.IS', 'PGSUS.IS', 'REEDR.IS', 'SASA.IS', 'SAHOL.IS', 'SDTTR.IS', 'SISE.IS', 'SKBNK.IS', 'SMRTG.IS', 'SOKM.IS', 'TAVHL.IS', 'TCELL.IS', 'THYAO.IS', 'TKFEN.IS', 'TOASO.IS', 'TSKB.IS', 'TTKOM.IS', 'TTRAK.IS', 'TUPRS.IS', 'ULKER.IS', 'VAKBN.IS', 'VESBE.IS', 'VESTL.IS', 'YKBNK.IS', 'ZOREN.IS']

TEFAS_FUND_TICKERS = [
    # örnek: 'AFT', 'TI2', 'IPB'
]

CONFIG = {
    'fx_tickers': ['USDTRY=X', 'EURTRY=X'],
    'commodity_usd_tickers': {
        'ALTIN_TRY': 'GC=F',
        'GUMUS_TRY': 'SI=F',
    },
    'salary_try': 25000,
    'target_test_months': 6,
    'target_lookback_days': 252,
    'download_period': '10y',
    'min_history_days': 180,
    'max_weight': 0.10,
    'gold_label': 'ALTIN_TRY',
    'gold_min': 0.05,
    'gold_max': 0.20,
    'use_llm_views': False,
    'llm_model': 'gpt-4.1-mini',
    'hybrid_alpha': 0.25,
    'strategies': ['Equal', 'HRP', 'BL', 'HYBRID'],
    'primary_data_source': 'openbb',
    'fallback_data_sources': ['yfinance'],
    'openbb_provider': None,
    'start_date': '2016-01-01',
    'max_missing_ratio': 0.20,
    'max_staleness_days': 14,
}
print('BIST ticker sayısı:', len(BIST_TICKERS))
print('TEFAS fon ticker sayısı:', len(TEFAS_FUND_TICKERS))


## 2) Helpers


In [ ]:
def try_fmt(x):
    if pd.isna(x):
        return '-'
    return f'₺{x:,.0f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

def pct_fmt(x):
    if pd.isna(x):
        return '-'
    return f'{x*100:.2f}%'

def style(fig, title):
    fig.update_layout(template='plotly_white', title=title, hovermode='x unified', legend_title_text='')
    return fig

def clean_returns(price_window):
    rets = price_window.pct_change()
    rets = rets.replace([np.inf, -np.inf], np.nan).dropna(how='any')
    if rets.empty:
        return rets
    valid_cols = [c for c in rets.columns if np.isfinite(rets[c]).all() and rets[c].std() > 1e-10]
    if len(valid_cols) == 0:
        return pd.DataFrame(index=rets.index)
    return rets[valid_cols]

def band_gold(weights, gold_label, gold_min, gold_max):
    w = weights.copy().astype(float)
    if gold_label not in w.index:
        return w / w.sum()
    current_gold = float(w[gold_label])
    target_gold = min(max(current_gold, gold_min), gold_max)
    if abs(target_gold - current_gold) < 1e-12:
        return w / w.sum()
    rest = w.drop(gold_label)
    if rest.sum() <= 0:
        w[:] = 0.0
        w[gold_label] = 1.0
        return w
    rest = rest / rest.sum()
    w[gold_label] = target_gold
    w.loc[rest.index] = (1 - target_gold) * rest
    return w / w.sum()

def normalize_weights(w, index):
    w = pd.Series(w).reindex(index).fillna(0.0).astype(float)
    s = w.sum()
    if s <= 0 or not np.isfinite(s):
        return pd.Series(1/len(index), index=index)
    return w / s


## 3) OpenBB-first data layer


In [ ]:
def coerce_openbb_output_to_series(output, symbol):
    if output is None:
        return None
    df = None
    if isinstance(output, pd.Series):
        s = output.copy()
        s.name = symbol
        return s
    if isinstance(output, pd.DataFrame):
        df = output.copy()
    elif hasattr(output, 'to_dataframe'):
        df = output.to_dataframe()
    elif hasattr(output, 'to_df'):
        df = output.to_df()
    elif hasattr(output, 'results'):
        try:
            df = pd.DataFrame(output.results)
        except Exception:
            df = None

    if df is None or len(df) == 0:
        return None

    if not isinstance(df.index, pd.DatetimeIndex):
        date_candidates = [c for c in df.columns if str(c).lower() in ['date', 'datetime', 'timestamp']]
        if date_candidates:
            df[date_candidates[0]] = pd.to_datetime(df[date_candidates[0]])
            df = df.set_index(date_candidates[0])

    close_candidates = [c for c in df.columns if str(c).lower() in ['close', 'adj_close', 'adjusted_close']]
    if len(close_candidates) == 0:
        numeric_cols = list(df.select_dtypes(include=[np.number]).columns)
        if len(numeric_cols) == 0:
            return None
        close_col = numeric_cols[0]
    else:
        close_col = close_candidates[0]

    s = pd.Series(df[close_col]).dropna()
    s.name = symbol
    return s if len(s) else None

def validate_series(s, symbol, config):
    issues = []
    if s is None or len(s) == 0:
        return None, ['empty']
    s = pd.Series(s).copy()
    s = s[~s.index.duplicated(keep='last')]
    s = s.sort_index()
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if not isinstance(s.index, pd.DatetimeIndex):
        try:
            s.index = pd.to_datetime(s.index)
        except Exception:
            issues.append('bad_index')
    if len(s) < config['min_history_days']:
        issues.append('short_history')
    if len(s) and (pd.Timestamp.today().normalize() - s.index.max().normalize()).days > config['max_staleness_days']:
        issues.append('stale')
    return s, issues

def fetch_series_from_yfinance(symbol, config):
    df = yf.download(symbol, period=config['download_period'], auto_adjust=True, progress=False)
    if df is None or len(df) == 0:
        return None
    if isinstance(df.columns, pd.MultiIndex):
        level0 = list(df.columns.get_level_values(0))
        close = df.xs('Close', axis=1, level=0) if 'Close' in level0 else df.iloc[:, :1]
    else:
        close = df[['Close']] if 'Close' in df.columns else df.iloc[:, :1]
    if isinstance(close, pd.DataFrame):
        if close.shape[1] == 0:
            return None
        close = close.iloc[:, 0]
    s = pd.Series(close).dropna()
    s.name = symbol
    return s if len(s) else None

def fetch_series_from_openbb(symbol, config):
    try:
        from openbb import obb
    except Exception:
        return None
    attempts = []
    if config['start_date'] is not None and config['openbb_provider']:
        attempts.append(lambda: obb.equity.price.historical(symbol, start_date=config['start_date'], provider=config['openbb_provider']))
    if config['start_date'] is not None:
        attempts.append(lambda: obb.equity.price.historical(symbol, start_date=config['start_date']))
    if config['openbb_provider']:
        attempts.append(lambda: obb.equity.price.historical(symbol, provider=config['openbb_provider']))
    attempts.append(lambda: obb.equity.price.historical(symbol))
    for fn in attempts:
        try:
            output = fn()
            s = coerce_openbb_output_to_series(output, symbol)
            if s is not None and len(s):
                return s
        except Exception:
            continue
    return None

def fetch_series(symbol, config):
    provider_order = [config['primary_data_source']] + [p for p in config['fallback_data_sources'] if p != config['primary_data_source']]
    errors = []
    for provider in provider_order:
        s = None
        if provider == 'openbb':
            s = fetch_series_from_openbb(symbol, config)
        elif provider == 'yfinance':
            s = fetch_series_from_yfinance(symbol, config)
        else:
            errors.append(f'unknown_provider:{provider}')
            continue
        s, issues = validate_series(s, symbol, config)
        if s is not None and len(s):
            return s, provider, issues
        errors.extend(issues)
    return None, None, list(dict.fromkeys(errors))

MANUAL_TEFAS_PRICES = None

def download_large_universe(config, bist_tickers, tefas_tickers):
    rows, downloaded = [], {}
    symbols = list(bist_tickers) + list(config['fx_tickers']) + list(config['commodity_usd_tickers'].values()) + list(tefas_tickers)
    for symbol in symbols:
        s, provider_used, issues = fetch_series(symbol, config)
        downloaded[symbol] = s
        rows.append({
            'source': symbol,
            'provider_used': provider_used,
            'n': 0 if s is None else len(s),
            'start': None if s is None else str(s.index.min().date()),
            'end': None if s is None else str(s.index.max().date()),
            'issues': ', '.join(issues) if issues else '',
            'status': 'ok' if s is not None and len(s) else 'empty',
        })
    return downloaded, pd.DataFrame(rows)

def build_synthetic_series(downloaded, config):
    synth = {}
    usdtry = downloaded.get('USDTRY=X')
    gold_usd = downloaded.get(config['commodity_usd_tickers']['ALTIN_TRY'])
    silver_usd = downloaded.get(config['commodity_usd_tickers']['GUMUS_TRY'])
    if gold_usd is not None and usdtry is not None:
        s = (gold_usd * usdtry).dropna()
        s.name = 'ALTIN_TRY'
        synth['ALTIN_TRY'] = s
    if silver_usd is not None and usdtry is not None:
        s = (silver_usd * usdtry).dropna()
        s.name = 'GUMUS_TRY'
        synth['GUMUS_TRY'] = s
    return synth

def prepare_large_universe_price_matrix(downloaded, synth_series, config, bist_tickers, tefas_tickers, manual_tefas_prices=None):
    series, rows = [], []
    for symbol in bist_tickers:
        s = downloaded.get(symbol)
        eligible = s is not None and len(s) >= config['min_history_days']
        if eligible:
            s = s.copy(); s.name = symbol.replace('.IS', ''); series.append(s)
        rows.append({'asset': symbol.replace('.IS', ''), 'n': 0 if s is None else len(s), 'eligible': bool(eligible), 'group': 'BIST'})
    for symbol in config['fx_tickers']:
        s = downloaded.get(symbol)
        eligible = s is not None and len(s) >= config['min_history_days']
        asset_name = symbol.replace('=X', '')
        if eligible:
            s = s.copy(); s.name = asset_name; series.append(s)
        rows.append({'asset': asset_name, 'n': 0 if s is None else len(s), 'eligible': bool(eligible), 'group': 'FX'})
    for name, s in synth_series.items():
        eligible = s is not None and len(s) >= config['min_history_days']
        if eligible:
            s = s.copy(); s.name = name; series.append(s)
        rows.append({'asset': name, 'n': 0 if s is None else len(s), 'eligible': bool(eligible), 'group': 'COMMODITY'})
    for symbol in tefas_tickers:
        s = downloaded.get(symbol)
        eligible = s is not None and len(s) >= config['min_history_days']
        if eligible:
            s = s.copy(); s.name = symbol; series.append(s)
        rows.append({'asset': symbol, 'n': 0 if s is None else len(s), 'eligible': bool(eligible), 'group': 'TEFAS'})
    if manual_tefas_prices is not None and len(manual_tefas_prices.columns) > 0:
        for col in manual_tefas_prices.columns:
            s = pd.Series(manual_tefas_prices[col]).dropna()
            eligible = len(s) >= config['min_history_days']
            if eligible:
                s.name = col; series.append(s)
            rows.append({'asset': col, 'n': len(s), 'eligible': bool(eligible), 'group': 'TEFAS_MANUAL'})
    eligible_assets = pd.DataFrame(rows)
    if len(series) < 5:
        raise ValueError('Geniş evrende yeterli tarihçesi olan varlık sayısı çok az.')
    prices = pd.concat(series, axis=1).sort_index().ffill()
    common_start = prices.apply(lambda s: s.first_valid_index()).max()
    prices = prices.loc[common_start:].dropna()
    monthly_points = prices.resample('MS').first().dropna()
    if len(monthly_points) < config['target_test_months'] + 1:
        raise ValueError(f"Geniş evrende yeterli ortak tarihçe yok. Aylık nokta sayısı: {len(monthly_points)}")
    lookback = min(config['target_lookback_days'], max(60, len(prices) - 40))
    months = config['target_test_months']
    return prices, eligible_assets, lookback, months


## 4) Views, strategies, backtest


In [ ]:
def views_from_prices(price_window):
    rets = clean_returns(price_window)
    if rets.empty or rets.shape[1] == 0:
        return {c: 0.0 for c in price_window.columns}
    ann = rets.mean() * 252
    vol = rets.std() * np.sqrt(252)
    score = (ann / vol.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0)
    score = score.reindex(price_window.columns).fillna(0.0)
    return score.clip(-0.15, 0.15).to_dict()

def get_views(price_window, config):
    return views_from_prices(price_window), 'rules'

def equal_weight(columns):
    return pd.Series(1 / len(columns), index=columns)

def inverse_vol_weight(price_window):
    rets = clean_returns(price_window)
    if rets.empty or rets.shape[1] == 0:
        return equal_weight(price_window.columns)
    iv = 1 / rets.std().replace(0, np.nan)
    iv = iv.replace([np.inf, -np.inf], np.nan).dropna()
    if iv.empty:
        return equal_weight(price_window.columns)
    w = pd.Series(0.0, index=price_window.columns)
    w.loc[iv.index] = iv / iv.sum()
    return normalize_weights(w, price_window.columns)

def hrp_weights_safe(price_window):
    rets = clean_returns(price_window)
    if rets.shape[0] < 2 or rets.shape[1] < 2:
        return inverse_vol_weight(price_window)
    try:
        hrp = HRPOpt(rets)
        w_sub = pd.Series(hrp.optimize()).reindex(rets.columns).fillna(0.0)
        w = pd.Series(0.0, index=price_window.columns)
        w.loc[w_sub.index] = w_sub
        return normalize_weights(w, price_window.columns)
    except Exception as e:
        print('HRP failed, inverse-vol kullanıldı:', str(e))
        return inverse_vol_weight(price_window)

def bl_weights_safe(price_window, config):
    rets = clean_returns(price_window)
    if rets.shape[0] < 2 or rets.shape[1] < 2:
        w = inverse_vol_weight(price_window)
        diag = pd.DataFrame({'asset': list(price_window.columns), 'view': [0.0]*len(price_window.columns), 'view_source': 'rules_insufficient_data'})
        return w, diag
    try:
        S = CovarianceShrinkage(rets, returns_data=True).ledoit_wolf()
        views, view_source = get_views(price_window, config)
        bl = BlackLittermanModel(S, pi='equal', absolute_views=views, omega='default')
        post = bl.bl_returns()
        ef = EfficientFrontier(post, S, weight_bounds=(0, config['max_weight']))
        ef.max_sharpe(risk_free_rate=0.0)
        w = pd.Series(ef.clean_weights()).reindex(price_window.columns).fillna(0.0)
        w = normalize_weights(w, price_window.columns)
        w = band_gold(w, config['gold_label'], config['gold_min'], config['gold_max'])
        diag = pd.DataFrame({'asset': list(views.keys()), 'view': list(views.values()), 'view_source': view_source})
        return w, diag
    except Exception as e:
        print('BL failed, inverse-vol kullanıldı:', str(e))
        w = inverse_vol_weight(price_window)
        fallback_views = views_from_prices(price_window)
        diag = pd.DataFrame({'asset': list(fallback_views.keys()), 'view': list(fallback_views.values()), 'view_source': 'rules_bl_fallback'})
        return w, diag

def hybrid_weights(price_window, config):
    w_hrp = hrp_weights_safe(price_window)
    w_bl, diag = bl_weights_safe(price_window, config)
    alpha = config['hybrid_alpha']
    w = (1 - alpha) * w_hrp + alpha * w_bl
    w = normalize_weights(w, price_window.columns)
    w = w.clip(lower=0, upper=config['max_weight'])
    w = normalize_weights(w, price_window.columns)
    w = band_gold(w, config['gold_label'], config['gold_min'], config['gold_max'])
    return w, diag

def compute_weights(strategy_name, price_window, config):
    if strategy_name == 'Equal':
        return equal_weight(price_window.columns), None
    if strategy_name == 'HRP':
        return hrp_weights_safe(price_window), None
    if strategy_name == 'BL':
        return bl_weights_safe(price_window, config)
    if strategy_name == 'HYBRID':
        return hybrid_weights(price_window, config)
    raise ValueError(f'Unknown strategy: {strategy_name}')

def get_rebalance_dates(prices, months):
    monthly = prices.resample('MS').first().dropna()
    month_starts = monthly.index[-months:]
    dates = []
    for dt in month_starts:
        idx = prices.index.searchsorted(dt)
        if idx < len(prices.index):
            dates.append(prices.index[idx])
    return pd.Index(pd.unique(pd.Index(dates)))

def run_strategy(strategy_name, prices, lookback, months, config):
    rebalance_dates = get_rebalance_dates(prices, months)
    cash = 0.0
    shares = pd.Series(0.0, index=prices.columns)
    hist, rebs, diags = [], [], []
    for i, dt in enumerate(rebalance_dates):
        cash += config['salary_try']
        window = prices.loc[:dt].tail(lookback)
        weights, diag = compute_weights(strategy_name, window, config)
        current = prices.loc[dt]
        total = float((shares * current).sum()) + cash
        shares = (weights * total) / current
        cash = 0.0
        nxt = rebalance_dates[i + 1] if i + 1 < len(rebalance_dates) else prices.index[-1]
        path = prices.loc[(prices.index >= dt) & (prices.index <= nxt)]
        values = path.mul(shares, axis=1).sum(axis=1)
        for t, v in values.items():
            hist.append({'date': t, 'strategy': strategy_name, 'portfolio_value': float(v)})
        row = {'date': dt, 'strategy': strategy_name, 'capital_after_contribution': total}
        for c, x in weights.items():
            row[f'weight_{c}'] = float(x)
        rebs.append(row)
        if diag is not None:
            tmp = diag.copy(); tmp['date'] = dt; tmp['strategy'] = strategy_name; diags.append(tmp)
    hist_df = pd.DataFrame(hist).drop_duplicates(['date', 'strategy'])
    weights_df = pd.DataFrame(rebs)
    diag_df = pd.concat(diags, ignore_index=True) if diags else pd.DataFrame()
    return hist_df, weights_df, diag_df

def run_all_strategies(prices, lookback, months, config):
    hist_parts, weight_parts, diag_parts = [], [], []
    for s in config['strategies']:
        h, w, d = run_strategy(s, prices, lookback, months, config)
        hist_parts.append(h); weight_parts.append(w)
        if not d.empty: diag_parts.append(d)
    hist_df = pd.concat(hist_parts, ignore_index=True)
    weights_df = pd.concat(weight_parts, ignore_index=True)
    diag_df = pd.concat(diag_parts, ignore_index=True) if diag_parts else pd.DataFrame()
    return hist_df, weights_df, diag_df


## 5) Reporting


In [ ]:
def build_summary(hist_df, months, config):
    rows = []
    contributed = config['salary_try'] * months
    for s in config['strategies']:
        sub = hist_df[hist_df['strategy'] == s].sort_values('date').copy()
        sub['ret'] = sub['portfolio_value'].pct_change().fillna(0.0)
        dd = sub['portfolio_value'] / sub['portfolio_value'].cummax() - 1
        rows.append({
            'Strategy': s,
            'Contributed': try_fmt(contributed),
            'Ending Value': try_fmt(sub['portfolio_value'].iloc[-1]),
            'Net Profit': try_fmt(sub['portfolio_value'].iloc[-1] - contributed),
            'TWR': pct_fmt(sub['portfolio_value'].iloc[-1] / contributed - 1),
            'Ann Vol': pct_fmt(sub['ret'].std() * np.sqrt(252)),
            'Sharpe': f"{{(sub['ret'].mean() / (sub['ret'].std() + 1e-9) * np.sqrt(252)):.2f}}",
            'MaxDD': pct_fmt(dd.min()),
        })
    return pd.DataFrame(rows)

def build_last_weights(weights_df):
    last = weights_df.sort_values('date').groupby('strategy').tail(1).copy()
    cols = [c for c in last.columns if c.startswith('weight_')]
    out = last[['strategy'] + cols].copy()
    out.columns = ['Strategy'] + [c.replace('weight_', '') for c in cols]
    for c in out.columns[1:]:
        out[c] = out[c].map(pct_fmt)
    return out.reset_index(drop=True)

def plot_market_structure(prices):
    fig = px.line(prices.tail(504) / prices.tail(504).iloc[0] * 100, x=prices.tail(504).index, y=prices.columns)
    style(fig, 'Son ~2 yıl normalize fiyatlar').show()

def plot_portfolio_values(hist_df):
    fig = px.line(hist_df, x='date', y='portfolio_value', color='strategy')
    fig.update_yaxes(tickprefix='₺', separatethousands=True)
    style(fig, 'Portföy değeri').show()


## 6) Run


In [ ]:
downloaded, coverage = download_large_universe(CONFIG, BIST_TICKERS, TEFAS_FUND_TICKERS)
synth_series = build_synthetic_series(downloaded, CONFIG)

for synth_name, synth_s in synth_series.items():
    coverage = pd.concat([
        coverage,
        pd.DataFrame([{
            'source': synth_name,
            'provider_used': 'synthetic',
            'n': len(synth_s),
            'start': str(synth_s.index.min().date()) if len(synth_s) else None,
            'end': str(synth_s.index.max().date()) if len(synth_s) else None,
            'issues': '',
            'status': 'ok' if len(synth_s) else 'empty',
        }])
    ], ignore_index=True)

display(coverage.sort_values(['status', 'source'], ascending=[False, True]))

prices, eligible_assets, lookback, months = prepare_large_universe_price_matrix(
    downloaded=downloaded,
    synth_series=synth_series,
    config=CONFIG,
    bist_tickers=BIST_TICKERS,
    tefas_tickers=TEFAS_FUND_TICKERS,
    manual_tefas_prices=MANUAL_TEFAS_PRICES,
)

display(eligible_assets)
print('Kullanılan varlık sayısı:', prices.shape[1])
print('Toplam gün:', len(prices), 'lookback:', lookback, 'test_months:', months)
display(prices.tail())

plot_market_structure(prices)

hist_df, weights_df, diag_df = run_all_strategies(prices, lookback, months, CONFIG)
summary = build_summary(hist_df, months, CONFIG)
display(summary)

plot_portfolio_values(hist_df)

last_weights = build_last_weights(weights_df)
display(last_weights.iloc[:, : min(15, last_weights.shape[1])])

if len(diag_df):
    diag_show = diag_df.copy()
    diag_show['view'] = diag_show['view'].map(pct_fmt)
    display(diag_show.tail(20))
